In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_distances

OUT = "figures"
os.makedirs(OUT, exist_ok=True)


In [ ]:
LIB = pd.read_csv("LIB.csv")
CORPUS = pd.read_csv("CORPUS.csv")
VOCAB = pd.read_csv("VOCAB.csv")

vocab_idx = VOCAB.set_index("term_str")[["idf", "stop", "max_pos_group"]]
tokens = CORPUS.merge(vocab_idx, left_on="term_str", right_index=True, how="left")

g = tokens.groupby("song_id")
TTR       = g["term_str"].agg(lambda s: s.nunique() / max(len(s), 1)).rename("TTR")
mean_idf  = g["idf"].mean().rename("mean_idf")
prop_stop = g["stop"].mean().rename("prop_stopwords")
prop_noun = g["max_pos_group"].apply(lambda s: (s == "N").mean()).rename("prop_nouns")
prop_verb = g["max_pos_group"].apply(lambda s: (s == "V").mean()).rename("prop_verbs")

feat = LIB.copy()
feat["year"]         = pd.to_datetime(feat["release_date"], errors="coerce").dt.year
feat["log_streams"]  = np.log10(feat["streams"].clip(lower=1))
feat["explicit_int"] = feat["explicit"].astype(int)
feat = (feat.merge(TTR, on="song_id", how="left")
            .merge(mean_idf, on="song_id", how="left")
            .merge(prop_stop, on="song_id", how="left")
            .merge(prop_noun, on="song_id", how="left")
            .merge(prop_verb, on="song_id", how="left"))

feat.head()


In [ ]:
num_cols = ["track_number", "duration_sec", "n_tokens", "log_streams", "year",
            "explicit_int", "TTR", "mean_idf", "prop_stopwords",
            "prop_nouns", "prop_verbs"]

corr = feat[num_cols].corr(method="spearman")

dist = 1 - corr.abs()
np.fill_diagonal(dist.values, 0.0)
condensed = squareform(dist.values, checks=False)
Z_feat = linkage(condensed, method="average")

cm = sns.clustermap(
    corr,
    row_linkage=Z_feat, col_linkage=Z_feat,
    cmap="RdBu_r", vmin=-1, vmax=1,
    annot=True, fmt=".2f",
    figsize=(10, 10),
    cbar_kws={"label": "Spearman ρ"},
    linewidths=0.4,
)
cm.fig.suptitle(
    "LIB feature correlations — hierarchically clustered\n"
    "(Spearman; rows/cols reordered by 1-|ρ| average linkage)",
    y=1.02, fontsize=12,
)
cm.savefig(f"{OUT}/01_clustered_correlation_heatmap.png",
           dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
docs = (CORPUS.sort_values(["song_id", "token_id"])
              .groupby("song_id")["term_str"]
              .apply(lambda s: " ".join(s.dropna().astype(str))))
docs = docs.reindex(LIB.song_id).fillna("")

vec  = TfidfVectorizer(min_df=2, max_df=0.9, sublinear_tf=True)
Xtf  = vec.fit_transform(docs.values)

svd  = TruncatedSVD(n_components=min(50, Xtf.shape[1] - 1), random_state=42)
Xs   = svd.fit_transform(Xtf)

tsne = TSNE(n_components=2, perplexity=15, init="pca",
            random_state=42, learning_rate="auto")
emb  = tsne.fit_transform(Xs)

plot_df = LIB[["song_id", "album", "explicit", "streams", "track_name"]].copy()
plot_df["x"]           = emb[:, 0]
plot_df["y"]           = emb[:, 1]
plot_df["log_streams"] = np.log10(plot_df["streams"].clip(lower=1))

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(12, 8))
palette = sns.color_palette("tab10", n_colors=plot_df["album"].nunique())
order   = plot_df.groupby("album")["song_id"].min().sort_values().index.tolist()

for i, alb in enumerate(order):
    sub   = plot_df[plot_df.album == alb]
    sizes = 30 + (sub["log_streams"] - plot_df["log_streams"].min()) * 40
    ax.scatter(
        sub["x"], sub["y"],
        s=sizes,
        color=palette[i],
        alpha=0.8,
        edgecolors=np.where(sub["explicit"], "black", "white"),
        linewidths=np.where(sub["explicit"], 1.4, 0.6),
        label=alb,
    )

for _, r in plot_df.nlargest(8, "streams").iterrows():
    ax.annotate(r["track_name"], (r["x"], r["y"]),
                fontsize=8, alpha=0.85,
                xytext=(4, 4), textcoords="offset points")

ax.set_title("t-SNE of songs over lyric TF-IDF\n"
             "hue = album, size = log streams, black border = explicit")
ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left",
          fontsize=8, title="Album")
plt.tight_layout()
plt.savefig(f"{OUT}/03_tsne_lyrics.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
docs_song  = (CORPUS.sort_values(["song_id", "token_id"])
                    .groupby("song_id")["term_str"]
                    .apply(lambda s: " ".join(s.dropna().astype(str)))).to_dict()
LIB["doc"] = LIB["song_id"].map(docs_song).fillna("")
album_doc  = LIB.groupby("album")["doc"].apply(" ".join).reset_index()

vec_a = TfidfVectorizer(min_df=2, max_df=0.95, sublinear_tf=True)
Xa    = vec_a.fit_transform(album_doc["doc"])

D = cosine_distances(Xa)
np.fill_diagonal(D, 0)
Z_alb = linkage(squareform(D, checks=False), method="average")

fig, ax = plt.subplots(figsize=(10, 5))
dendrogram(
    Z_alb,
    labels=album_doc["album"].tolist(),
    leaf_rotation=30, leaf_font_size=9,
    color_threshold=0.6 * max(Z_alb[:, 2]),
    ax=ax,
)
ax.set_title("Hierarchical clustering of albums by lyrical similarity\n"
             "(cosine distance over TF-IDF, average linkage)")
ax.set_ylabel("Cosine distance")
plt.tight_layout()
plt.savefig(f"{OUT}/05_album_dendrogram.png", dpi=150, bbox_inches="tight")
plt.show()
